# LoRA judge for the fact-checker (Google Colab, free T4 GPU)

Fine-tunes **Qwen3-4B-Instruct-2507** (Apache-2.0) with LoRA to map *claim + evidence -> TRUE / FALSE / NOT_ENOUGH_INFO*,
then exports a **GGUF** file to run locally in **Ollama**.

1. Runtime -> Change runtime type -> **T4 GPU**.
2. On your machine run `make lora-data`, then upload `data/lora/train.jsonl` and `data/lora/val.jsonl` when asked below.
3. Run all cells (~30-60 min). Download `factjudge-Q4_K_M.gguf` at the end into `1-fact-checker/lora/`.
4. On your machine: `make ollama-judge`, then `make judge-eval`.

The prompt format comes from `factcheck/judge_lora.py` and is already inside the uploaded data - do not change it here.
If an Unsloth call fails, check the current Unsloth Qwen3 notebook for API changes.

In [ ]:
%%capture
!pip install unsloth

In [ ]:
from google.colab import files
uploaded = files.upload()   # choose train.jsonl and val.jsonl
assert {"train.jsonl", "val.jsonl"} <= set(uploaded), uploaded.keys()

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_LEN = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=MAX_LEN,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407,
)

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": "train.jsonl", "val": "val.jsonl"})

def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False) for m in batch["messages"]]}

ds = ds.map(to_text, batched=True)
print(ds)
print(ds["train"][0]["text"][:1500])

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_LEN,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)
# learn only the answer, not the prompt (Qwen ChatML markers)
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")
stats = trainer.train()
stats

## Quick check on the validation set (the real comparison is `make judge-eval` on your machine)

In [ ]:
import json, re
from collections import Counter

FastLanguageModel.for_inference(model)
correct, preds = 0, Counter()
val = ds["val"]
for ex in val:
    msgs = ex["messages"][:2]  # system + user
    inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=16, do_sample=False)
    text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    m = re.search(r"TRUE|FALSE|NOT_ENOUGH_INFO", text)
    pred = m.group(0) if m else "ERROR"
    preds[pred] += 1
    correct += pred == ex["label"]
print(f"val accuracy: {correct / len(val):.1%}", dict(preds))

## Export to GGUF for Ollama

In [ ]:
model.save_pretrained_gguf("factjudge", tokenizer, quantization_method="q4_k_m")
import glob, shutil
gguf = sorted(glob.glob("factjudge*/*.gguf") + glob.glob("*.gguf"))
print(gguf)
shutil.copy(gguf[-1], "factjudge-Q4_K_M.gguf")
files.download("factjudge-Q4_K_M.gguf")